# 02 — EMIT Preprocessing

Convert the EMIT reflectance cube to a georeferenced raster format for later spatial analysis.

This notebook follows the user's existing workflow using the EMIT reflectance variable and the file geotransform.  
For products that contain a GLT/location group, a separate GLT-based orthorectification workflow is kept below.

In [ ]:
from pathlib import Path
import xarray as xr
import numpy as np
import rasterio
from affine import Affine

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMIT_NC = DATA_DIR / "EMIT_L2A_RFL_example.nc"
OUT_TIF = OUTPUT_DIR / "EMIT_reflectance.tif"

if not EMIT_NC.exists():
    raise FileNotFoundError(f"Missing input: {EMIT_NC.resolve()}")

ds = xr.open_dataset(EMIT_NC)
rfl = ds["reflectance"].values

# Existing EMIT workflow assumes (rows, cols, bands).
if rfl.ndim != 3:
    raise ValueError(f"Expected a 3-D reflectance cube, got {rfl.shape}")

cube = np.transpose(rfl, (2, 0, 1)).astype("float32")
print("Cube shape (bands, rows, cols):", cube.shape)

In [ ]:
if "geotransform" not in ds.attrs:
    raise KeyError("The EMIT file does not expose 'geotransform' in ds.attrs.")

transform = Affine.from_gdal(*ds.attrs["geotransform"])
print("Affine transform:", transform)

# The original workflow used EPSG:4326 for this exported raster.
crs = "EPSG:4326"

with rasterio.open(
    OUT_TIF,
    "w",
    driver="GTiff",
    height=cube.shape[1],
    width=cube.shape[2],
    count=cube.shape[0],
    dtype="float32",
    crs=crs,
    transform=transform,
    interleave="band",
) as dst:
    dst.write(cube)

print("Saved:", OUT_TIF.resolve())

## Optional GLT-based export

The original work also opened the `location` group and used `glt_x` and `glt_y` to remap the reflectance cube.

Use that route only when the EMIT product contains the required GLT variables and when the spatial geometry needs this remapping.

In [ ]:
# Optional GLT workflow. Run only for products that contain the required variables.

try:
    loc = xr.open_dataset(EMIT_NC, group="location")
    print("Location variables:", list(loc.data_vars))

    if "glt_x" in loc and "glt_y" in loc:
        glt_x = loc["glt_x"].values.astype(int)
        glt_y = loc["glt_y"].values.astype(int)

        valid = (glt_x > 0) & (glt_y > 0)
        glt_x[valid] -= 1
        glt_y[valid] -= 1

        out_rows, out_cols = glt_x.shape
        bands = rfl.shape[2]
        ortho = np.zeros((bands, out_rows, out_cols), dtype=np.float32)

        for b in range(bands):
            band = rfl[:, :, b]
            temp = np.zeros((out_rows, out_cols), dtype=np.float32)
            temp[valid] = band[glt_y[valid], glt_x[valid]]
            ortho[b] = temp

        print("GLT-remapped cube:", ortho.shape)
    else:
        print("glt_x/glt_y not found; skip GLT workflow.")
except Exception as exc:
    print("Optional GLT step skipped:", exc)